In [17]:
import pandas as pd
import numpy as np

Units = pd.read_csv('./Units.csv')
Lines = pd.read_csv('./Lines.csv')
Links = pd.read_csv('./Links.csv')
Transformers = pd.read_csv('./Transformers.csv')
Buses = pd.read_csv('./Buses.csv') 
Loads = pd.read_csv('./Loads.csv')

file_path = "./GB_System_1week.xlsx"

National_Data = pd.read_excel('/Users/zm348/PhD/Projects/Nature-EV/data/Inputs_prepared/GB_System_1week.xlsx', sheet_name='national_data')
Loads

,LoadID,bus_name,load_weight,RegionName
0,1,way/92419253-275,0.002569,Macclesfield
1,2,way/1190395478-220,0.000261,Strichen
2,3,way/262523325-400,0.000241,Dunbar
3,4,way/49499923-400,0.004041,Hams Hall
4,5,way/25935453-400,0.003187,Burwell Main
...,...,...,...,...
380,381,GB71-275,0.001419,Upper Boat
381,382,GB52-400,0.005545,NaN
382,383,way/87464485-400,0.004038,Rye House
383,384,way/87464485-400,0.004038,Rye House


## Initialize

In [18]:
# with pd.ExcelWriter(file_path, mode='a', engine='openpyxl') as writer:
    # Lines.to_excel(writer, sheet_name='Lines', index=False)
    # Units.to_excel(writer, sheet_name='Units', index=False)
    # Loads.to_excel(writer, sheet_name='Demands', index=False)
    # Buses.to_excel(writer, sheet_name='Buses', index=False)
    # National_Data.to_excel(writer, sheet_name='national_data', index=False)
    # Transformers.to_excel(writer, sheet_name='Transformers', index=False)
    # Links.to_excel(writer, sheet_name='Links', index=False)

In [2]:
GB_System = pd.read_excel(file_path, sheet_name=None)

## Bus ID Mapping

In [3]:
Buses['BusID'] = range(len(Buses))
Busname_to_BusID = dict(zip(Buses['bus_name'], Buses['BusID']))

## line

In [4]:
Lines.rename(columns={'LineID': 'Line_ID'}, inplace=True)
Lines['NodeIn'] = Lines['bus0'].map(Busname_to_BusID)
Lines['NodeOut'] = Lines['bus1'].map(Busname_to_BusID)
Lines.rename(columns={'capacity': 'cap'}, inplace=True)
# Lines.rename(columns={'x': 'xl'}, inplace=True)

Links.rename(columns={'LinkID': 'Line_ID'}, inplace=True)
Links['NodeIn'] = Links['bus0'].map(Busname_to_BusID)
Links['NodeOut'] = Links['bus1'].map(Busname_to_BusID)
Links.rename(columns={'capacity': 'cap'}, inplace=True)
# Links.rename(columns={'x': 'xl'}, inplace=True)

Transformers.rename(columns={'TransformerID': 'Line_ID'}, inplace=True)
Transformers['NodeIn'] = Transformers['bus0'].map(Busname_to_BusID)
Transformers['NodeOut'] = Transformers['bus1'].map(Busname_to_BusID)
Transformers.rename(columns={'s_nom': 'cap'}, inplace=True)

line = pd.concat([Lines, Links, Transformers], ignore_index=True)
line = line[['Line_ID', 'NodeIn', 'NodeOut', 'cap', 'x', 'voltage']]
line['Line_ID'] = range(len(line))
line

,Line_ID,NodeIn,NodeOut,cap,x,voltage
0,0,175,419.0,921.668,14.577213,275.0
1,1,99,88.0,3574.953,3.447219,400.0
2,2,118,35.0,1843.335,0.310518,275.0
3,3,474,128.0,3574.953,3.492045,400.0
4,4,139,74.0,1474.668,7.949573,220.0
...,...,...,...,...,...,...
735,735,372,NaN,NaN,NaN,NaN
736,736,376,216.0,NaN,NaN,NaN
737,737,386,NaN,NaN,NaN,NaN
738,738,388,315.0,NaN,NaN,NaN


In [5]:
line.loc[line['NodeOut'].isna(), 'NodeOut'] = line.loc[line['NodeOut'].isna(), 'NodeIn']
line['cap'] = line['cap'].fillna(line['cap'].mean())
line['x'] = line['x'].fillna(0.01)
line['voltage'] = line['voltage'].fillna(400)
# p.u.
Sbase = 1000 # MVA
line['xl'] = line['x']/ (line['voltage'] * line['voltage'] / Sbase)
line = line[['Line_ID', 'NodeIn', 'NodeOut', 'cap', 'xl']]
# line['xl'] = line['xl']/Xbase

## Loads Mapping

In [6]:
Loads_merged = Loads.groupby('bus_name', as_index=False)['load_weight'].sum()
BusID_to_Loads = dict(zip(Loads_merged['bus_name'].map(Busname_to_BusID), Loads_merged['load_weight']))

## dem

In [7]:
dem = pd.DataFrame()
dem['Bus_ID'] = Buses['BusID']
dem['No'] = range(1, len(dem)+1)
dem['base'] = dem['Bus_ID'].map(BusID_to_Loads)
dem['base'] = dem['base'].fillna(0)
dem

,Bus_ID,No,base
0,0,1,0.000000
1,1,2,0.000426
2,2,3,0.000000
3,3,4,0.000000
4,4,5,0.000407
...,...,...,...
539,539,540,0.000000
540,540,541,0.000000
541,541,542,0.000000
542,542,543,0.000000


In [8]:
mask = National_Data.iloc[:, 0] == 'Demand'
demand_dict = National_Data.loc[mask].iloc[0, 1:].to_dict()

for t_col in demand_dict.keys():
    dem[t_col] = dem['base'] * demand_dict[t_col]

/var/folders/f0/zj71chhd6pvdlx6llbr36gtr0000gp/T/ipykernel_24753/699887200.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dem[t_col] = dem['base'] * demand_dict[t_col]
/var/folders/f0/zj71chhd6pvdlx6llbr36gtr0000gp/T/ipykernel_24753/699887200.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dem[t_col] = dem['base'] * demand_dict[t_col]
/var/folders/f0/zj71chhd6pvdlx6llbr36gtr0000gp/T/ipykernel_24753/699887200.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` m

## dg (At the end, the solar and wind have been neglected)

In [9]:
Units = Units[Units['Status'] == 'operating']
Units['Bus_ID'] = Units['Bus name'].map(Busname_to_BusID)

In [10]:
Units = Units[['Technology', 'Bus_ID', 'capacity']]
Units = Units.groupby(['Technology', 'Bus_ID'], as_index=False)['capacity'].sum()
dg = Units.sort_values(by=['Bus_ID', 'Technology']).reset_index(drop=True)

dg['NO'] = range(1, len(dg) + 1)
dg.rename(columns={'Technology': 'Type', 'capacity': 'Pmax'}, inplace=True)

In [11]:
tech_costs = {
    'solar':   0.0,
    'onwind':  0.0,
    'offwind': 0.0,
    'hydro':   0.0,     
    'nuclear': 10.0,    
    'biomass': 40.0,    
    'gas':     45.0,    
    'coal':    30.0,    
    'oil':     130.0
}


carbon_intensity = {
    'biomass': 0,    
    'solar': 0,      
    'offwind': 0,    
    'onwind': 0,     
    'nuclear': 0,    
    'hydro': 0,      
    'coal': 820,     
    'oil': 650,      
    'gas': 490       
}

dg['lambdaG'] = dg['Type'].map(tech_costs)
dg['Carbon'] = dg['Type'].map(carbon_intensity)

# 添加噪声
np.random.seed(42)  # 固定随机种子，保证复现
dg['lambdaG'] *= (1 + np.random.normal(0, 0.05, size=len(dg)))  # ±5% 波动
dg['Carbon']  *= (1 + np.random.normal(0, 0.02, size=len(dg)))  # ±2% 波动



## res

In [12]:
res = dg.copy()
res['base'] = res.groupby('Type')['Pmax'].transform(lambda x: x / x.sum())
res = res[['Bus_ID','base','Type']]
res.rename(columns={'Type': 'NO'}, inplace=True)
res = res[res['NO'].isin(['onwind', 'offwind', 'solar'])]
res

,Bus_ID,base,NO
0,0,0.000726,offwind
1,0,0.004365,onwind
3,1,0.005675,onwind
4,1,0.000506,solar
5,4,0.011950,onwind
...,...,...,...
503,486,0.000890,solar
505,488,0.014787,onwind
508,489,0.008997,offwind
510,490,0.033169,onwind


In [13]:
mask_wind = National_Data.iloc[:, 0] == 'Wind'
mask_solar = National_Data.iloc[:, 0] == 'Solar'
demand_dict_wind = National_Data.loc[mask_wind].iloc[0, 1:].to_dict()
demand_dict_solar = National_Data.loc[mask_solar].iloc[0, 1:].to_dict()

for t_col in demand_dict_wind.keys():  # shared keys
    res[t_col] = 0.0
    res.loc[res['NO'].isin(['onwind', 'offwind']), t_col] = \
        res['base'] * demand_dict_wind[t_col] / 2 # Divide by 2 for onwind and offwind
    res.loc[res['NO'] == 'solar', t_col] = \
        res['base'] * demand_dict_solar[t_col]

order = ['onwind', 'offwind', 'solar']  # 自定义顺序
res['NO'] = pd.Categorical(res['NO'], categories=order, ordered=True)

res = res.sort_values(by=['NO', 'Bus_ID'], ascending=[True, True]).reset_index(drop=True)

# total_T5_solar = res.loc[res['NO'] == 'solar', 'T5'].sum()
# print(total_T0_solar)
# total_T0_wind = res.loc[res['NO'] == 'onwind', 'T0'].sum()
# print(total_T0_wind)
res

/var/folders/f0/zj71chhd6pvdlx6llbr36gtr0000gp/T/ipykernel_24753/12355363.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  res[t_col] = 0.0
/var/folders/f0/zj71chhd6pvdlx6llbr36gtr0000gp/T/ipykernel_24753/12355363.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  res[t_col] = 0.0
/var/folders/f0/zj71chhd6pvdlx6llbr36gtr0000gp/T/ipykernel_24753/12355363.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all 

,Bus_ID,base,NO,T1,T2,T3,T4,T5,T6,T7,...,T325,T326,T327,T328,T329,T330,T331,T332,T333,T334
0,0,0.004365,onwind,6.250362,6.687411,7.127454,7.738819,8.438881,10.435460,12.842404,...,11.467015,11.323863,12.096019,13.027496,1.338616e+01,13.339992,13.029391,13.074958,13.470081,13.623046
1,1,0.005675,onwind,8.126542,8.694781,9.266913,10.061792,10.971993,13.567888,16.697328,...,14.909087,14.722964,15.726900,16.937980,1.740431e+01,17.344278,16.940443,16.999688,17.513415,17.712297
2,4,0.011950,onwind,17.110767,18.307217,19.511865,21.185514,23.101981,28.567743,35.156907,...,31.391692,30.999803,33.113630,35.663609,3.664548e+01,36.519087,35.668796,35.793538,36.875212,37.293965
3,7,0.002022,onwind,2.894679,3.097086,3.300879,3.584015,3.908230,4.832889,5.947597,...,5.310625,5.244327,5.601930,6.033317,6.199423e+00,6.178041,6.034195,6.055298,6.238288,6.309129
4,8,0.000749,onwind,1.072103,1.147069,1.222548,1.327413,1.447493,1.789959,2.202814,...,1.966898,1.942344,2.074789,2.234562,2.296083e+00,2.288163,2.234887,2.242703,2.310477,2.336715
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
381,480,0.001365,solar,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,3.067994,1.811297,0.737715,0.131879,2.331883e-06,0.000000,0.000000,0.000000,0.000000,0.000000
382,481,0.006645,solar,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,14.930906,8.814979,3.590212,0.641810,1.134850e-05,0.000000,0.000000,0.000000,0.000000,0.000000
383,482,0.000283,solar,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.636325,0.375676,0.153007,0.027353,4.836498e-07,0.000000,0.000000,0.000000,0.000000,0.000000
384,486,0.000890,solar,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.999878,1.180697,0.480881,0.085965,1.520042e-06,0.000000,0.000000,0.000000,0.000000,0.000000


In [14]:
dg = dg[~dg['Type'].str.contains('wind|solar', case=False, na=False)].copy()

# Write datas into xlsx

In [15]:
with pd.ExcelWriter('GB_System.xlsx', mode='a', if_sheet_exists='replace', engine='openpyxl') as writer:
    line.to_excel(writer, sheet_name='line', index=False)
    dem.to_excel(writer, sheet_name='dem', index=False)
    res.to_excel(writer, sheet_name='res', index=False)
    dg.to_excel(writer, sheet_name='dg', index=False)

